In [21]:
!pip install youtube-transcript-api



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
load_dotenv()
from langchain_core.runnables import RunnablePassthrough,Runnable,RunnableBranch,RunnableSequence,RunnableLambda,RunnableParallel


Fetching Youtube Transcripts

In [23]:

ytt_api = YouTubeTranscriptApi()
video_id='7ARBJQn6QkM'
fetched_transcript=ytt_api.fetch(video_id,languages=['en'])

text=""
for snippet in fetched_transcript:
    text+=(snippet.text)

# indexable
last_snippet = fetched_transcript[-1]

# provides a length
snippet_count = len(fetched_transcript)
print(snippet_count)

IpBlocked: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=7ARBJQn6QkM! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

Ways to work around this are explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).


If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide th

In [ ]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
chunks = splitter.create_documents([text])
# for i,chunk in enumerate(chunks):
#     print('chunk number is ',i)
#     print(chunk)

embedding=GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-2'
)
vector_store = Chroma(
    embedding_function=embedding,
    collection_name='yt-transcripts',
)

vector_store.add_documents(chunks)
    

['e8f0fac5-9c91-46ec-af87-05542caa362e',
 'f2b38014-9de3-403f-a83d-f0962a91c589',
 '014ea9c8-c3fa-4741-8c9c-8a9ddff2fd0e',
 '7343c42a-bc5b-469f-aa77-4c962f372bbc',
 '0a73cec9-7dd1-4862-8854-f651c64241f2',
 '747cba5e-4721-4b50-ba1f-b80cda7eb63f',
 '3332bbd0-8dd5-4eb5-b310-239d1b5b6d0a',
 'f0572e53-4f26-446e-9eda-fcc5764cb6e7',
 '6ae9e87d-61f0-4d95-8e25-d3d192578144',
 '22985a06-9e5a-4b35-81a8-ab309c7fd67a',
 '5953001f-6ac4-45b2-99f0-d9fac7012adc',
 '9c3288cd-02fd-4d37-9006-990e516b46da',
 'b741e3ec-f385-48fe-a79d-5d85ac037254',
 '583c565f-5f0f-49ef-930f-07e69e811217',
 '9a3ba5d4-1b51-42fd-b544-8291bbae02d4',
 '71081fd1-112e-4192-b6e6-6e15fd143da2',
 '477a4b9b-82b3-4188-868d-76b2f4df2cb4',
 'b561b337-e175-4e9e-8873-9c292d802773',
 '75a47c01-dd29-43da-adb2-42b467788aed',
 'f78eaf6c-47f7-42d4-aa57-e736f054ab8a',
 'ae32a403-d124-4d86-bfb3-1df50ef4ca75',
 'f36e0f31-945a-4735-a5fc-d42a3bed3d7b',
 'ff27bdd4-9724-483b-856a-a2161240febd',
 '7cc6cefb-fc01-49a6-b9be-84c34643ccec',
 'b440d801-6a4c-

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever.invoke('what is the future look like in a decade from now')

[Document(id='282cf8c8-af87-46ed-862f-88eea6bdf940', metadata={}, page_content="new economies and new capabilities, new economies.What would happen if a video conference made\xa0\nit possible for us to see each other withouthaving to travel anymore? All of a sudden\xa0\nit's actually okay to work further away fromhome and from work, work and live\xa0\nfurther away. And so you ask yourself kind ofthese questions. You know what would happen\xa0\nif I have a software programmer with meall the time and whatever it is I can dream up,\xa0\nthe software programmer could write for me. Youknow what would, what would happen\xa0\nif I just had a seed of an idea andand I rough it out and all of sudden a you know\xa0\na prototype of a production was put in frontof me? And what how would that change my life and\xa0\nhow would that change my opportunity? And youknow what does it free me to be able to do and\xa0\nand so on so forth. And so I think that the nextthe next decade intelligence, not for eve

Augmentation


In [37]:
llm=ChatGoogleGenerativeAI(
    model='gemini-3.7-flash',
    temperature=0.7
)

prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [27]:
query='what is the future look like in a decade from now'
res=retriever.invoke(query)
context=""
for chunk in res:
    context+=chunk.page_content
print(context)

new economies and new capabilities, new economies.What would happen if a video conference made 
it possible for us to see each other withouthaving to travel anymore? All of a sudden 
it's actually okay to work further away fromhome and from work, work and live 
further away. And so you ask yourself kind ofthese questions. You know what would happen 
if I have a software programmer with meall the time and whatever it is I can dream up, 
the software programmer could write for me. Youknow what would, what would happen 
if I just had a seed of an idea andand I rough it out and all of sudden a you know 
a prototype of a production was put in frontof me? And what how would that change my life and 
how would that change my opportunity? And youknow what does it free me to be able to do and 
and so on so forth. And so I think that the nextthe next decade intelligence, not for everything 
but for for some things, would basically becomesuperhuman. But I can tellit opened up this universe of oppo

In [29]:
final_prompt = prompt.invoke({"context": context, "question": query})

In [ ]:
parser = StrOutputParser()
print(final_prompt)

text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      new economies and new capabilities, new economies.What would happen if a video conference made\xa0\nit possible for us to see each other withouthaving to travel anymore? All of a sudden\xa0\nit's actually okay to work further away fromhome and from work, work and live\xa0\nfurther away. And so you ask yourself kind ofthese questions. You know what would happen\xa0\nif I have a software programmer with meall the time and whatever it is I can dream up,\xa0\nthe software programmer could write for me. Youknow what would, what would happen\xa0\nif I just had a seed of an idea andand I rough it out and all of sudden a you know\xa0\na prototype of a production was put in frontof me? And what how would that change my life and\xa0\nhow would that change my opportunity? And youknow what does it free me to be able to do and\x

In [39]:
ans = llm.invoke(final_prompt)

# Extract the string from the content list
text_output = ans.content[0]['text']
print(text_output)

Based on the provided transcript, the next decade is expected to look like the following:

* **Superhuman Intelligence:** Intelligence will basically become "superhuman" for some things (though not for everything).
* **The Application Science of AI:** While the last 10 years focused on the science of AI, the next 10 years will focus on applied research and the "application science of AI."
* **Broad Industry Applications:** AI will be applied to fields such as:
  * Digital biology
  * Climate technology (including accurately predicting weather patterns within a kilometer)
  * Agriculture and fishery
  * Robotics
  * Transportation and optimizing logistics
* **Enormous Change and New Opportunities:** It is described as being on the cusp of truly enormous change, opening up a "universe of opportunities and universe of problems that we can go solve."


In [41]:
def format_docs(res):
  context_text = "\n\n".join(doc.page_content for doc in res)
  return context_text

parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [42]:
parallel_chain.invoke('what will 20 years from now look like')


{'context': "new economies and new capabilities, new economies.What would happen if a video conference made\xa0\nit possible for us to see each other withouthaving to travel anymore? All of a sudden\xa0\nit's actually okay to work further away fromhome and from work, work and live\xa0\nfurther away. And so you ask yourself kind ofthese questions. You know what would happen\xa0\nif I have a software programmer with meall the time and whatever it is I can dream up,\xa0\nthe software programmer could write for me. Youknow what would, what would happen\xa0\nif I just had a seed of an idea andand I rough it out and all of sudden a you know\xa0\na prototype of a production was put in frontof me? And what how would that change my life and\xa0\nhow would that change my opportunity? And youknow what does it free me to be able to do and\xa0\nand so on so forth. And so I think that the nextthe next decade intelligence, not for everything\xa0\nbut for for some things, would basically becomesuperhu

In [46]:
main_chain = parallel_chain | prompt | llm | parser
ans = main_chain.invoke("Can you summarize the video")
print(ans)

TypeError: BaseModel.__init__() takes 1 positional argument but 2 were given